# Examples on how to access and use the seismic benchmark dataset hosted on the GeoNet Open Data Bucket

The benchmark dataset is hosted on `s3://geonet-open-data/seismic-products/benchmark-dataset/`

The dataset is structure is:

```
benchmark-dataset/
├── event_dataset/
│   ├── metadata/
│   │   └── metadata_events_{2013..2024}.csv
│   └── waveform_data/
│       ├── counts/
│       │   └── waveforms_counts_{2013..2024}.h5
│       └── units/
│           └── waveforms_units_{2013..2024}.h5
└── noise_dataset/
    ├── metadata/
    │   └── metadata_noise.csv
    └── waveform_data/
        ├── counts/
        │   └── waveforms_counts_noise.h5
        └── units/
            └── waveforms_units_noise.h5
```


Examples include:
- Viewing the content and downloading using AWS CLI (`https://docs.aws.amazon.com/cli/`). 
- Reading the dataset locally after downloading (using AWS CLI) 


# Bash commands to view the content and download. 
Please install AWS CLI in your system before using the commands below. The commands have been appended with a ! in the notebook cells to indicate that they are bash commands. If you want to use these commands in a separate terminal window, make sure to remove the ! at the beginning. 

1. View the contents

In [ ]:
!aws s3 ls --no-sign-request  --recursive --human-readable s3://geonet-open-data/seismic-products/benchmark-dataset/

2. Download a file into the current directory

In [ ]:
!aws s3 cp --no-sign-request s3://geonet-open-data/seismic-products/benchmark-dataset/event_dataset/metadata/metadata_events_2013.csv .

3. Download a specific directory into the current current directory

In [ ]:
!aws s3 sync --no-sign-request s3://geonet-open-data/seismic-products/benchmark-dataset/event_dataset/metadata/ . 

4. Download the full dataset in to the current directory. Please make sure you have sufficient disk space (about 1.5 TB) if you want to do it. 

In [ ]:
!aws s3 sync --no-sign-request s3://geonet-open-data/seismic-products/benchmark-dataset/ . 

# Read the locally downloaded data, CSV and HDF5

1. Read an event metadata CSV file into a pandas dataframe and display the first 5 rows of the dataframe.


In [ ]:
import pandas as pd
df = pd.read_csv('metadata_events_2013.csv')
df.head()

2. Merge multiple CSV files into a single pandas dataframe and display the first 5 rows of the merged dataframe.

In [ ]:
import pandas as pd 

csv_files = ["metadata_events_2013", "metadata_events_2016"] # add more CSV files to the list as needed

df = pd.concat([pd.read_csv(f"{file}.csv") for file in csv_files], ignore_index=True)
df.head()

3. Read downloaded HDF file

In [ ]:
import h5py
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt 


h5_file = Path("/seis/prj/waveform/GEONET_BENCHMARK_WAVEFORM_DATASET/DATASET/event_dataset/waveform_data/counts/waveforms_counts_2013.h5")


with h5py.File(h5_file, "r") as f:
    
    # read the contents of the "data_format" group
    data_format_group = f["data_format"]
    for (key, val) in data_format_group.items():
        print(key, val[()])
        
    # data group: one dataset per trace, metadata stored in attrs
    data_group = f["data"]
    
    # total number of traces in the file
    num_traces = len(data_group)
    print(f"Total number of traces: {num_traces}")
    
    # list first 3 trace names
    trace_names = list(data_group.keys())
    print(f"First 3 trace names: {trace_names[:3]}")
    
    # read one specific trace and its metadata
    trace_name = "2013p866641_NZ_CKHZ_10_EH"
    if trace_name not in data_group:
        raise KeyError(f"{trace_name} not found in {h5_file.name}")
    
    trace_dataset = data_group[trace_name]
    print(trace_dataset)
    
    # read the waveform data and metadata
    waveform_data = trace_dataset[()]
    print (waveform_data)
    waveform_metadata = dict(trace_dataset.attrs)    
    print (waveform_metadata)

    # plot the waveform data
    taxis = np.arange(len(waveform_data[0])) / waveform_metadata["trace_sampling_rate_Hz"]
    fig, axes = plt.subplots(figsize=(10, 5), nrows=3, ncols=1, sharex=True)
    
    for a, ax in enumerate(axes):
        ax.plot(taxis, waveform_data[a], color="k")
        ax.text(0.02, 0.9, waveform_metadata["station_channels"][a], transform=ax.transAxes, fontsize=10, verticalalignment='top')


        if waveform_metadata["trace_P_arrival_sample"] is not None:
            ax.axvline(x=waveform_metadata["trace_P_arrival_sample"] / waveform_metadata["trace_sampling_rate_Hz"], color="red", linestyle="--", label="P-wave arrival")
    
        if waveform_metadata["trace_S_arrival_sample"] != 'None':
            ax.axvline(x=waveform_metadata["trace_S_arrival_sample"] / waveform_metadata["trace_sampling_rate_Hz"], color="green", linestyle="--", label="S-wave arrival")
    
        ax.set_ylabel("Counts")
        
    axes[2].set_xlabel("Time (s)")
    fig.suptitle(f"{trace_name}")
    plt.show()